# 프로젝트 4 - Weekend 2: 멀티모달 추출 + RAG 베이스라인

**이번 주말 목표** — 비정형 데이터의 **모든 모달리티**(PDF 표/이미지/차트, 오디오, 비디오)에서 정보를 끌어내고, **RAG 베이스라인**까지 완성한다. Weekend 3는 이 RAG를 Graph-RAG로 보강.

**학습 목표**:
1. `PyMuPDF`로 PDF 페이지 → 이미지, 표 영역 탐지·추출
2. `pandas` DataFrame과 Markdown 표로 변환·정제
3. GPT-4o Vision API로 이미지 캡셔닝과 차트 데이터 복원(Pydantic 구조화)
4. **Whisper**(LangChain `OpenAIWhisperParser`)로 오디오 → 텍스트, 청크 단위 Document
5. **OpenCV**로 비디오 프레임 샘플링, **ffmpeg**로 오디오 트랙 분리 후 transcribe
6. PDF·오디오·비디오를 단일 `Document` 스키마로 통합
7. **FAISS** 멀티모달 인덱싱 + `element_type` 가중치 검색
8. 멀티모달 컨텍스트 RAG 답변 + **LLM-as-Judge**로 충실도 검증


> 📦 실습 데이터:
> - PDF: `data/quarterly_report.pdf`, `data/research_paper.pdf` (Weekend 1과 동일)
> - 오디오: `data/audio/*.mp3` 3개 (회의 — 매출/제품/채용)
> - 비디오: `data/video/earnings_briefing.mp4` (실적 브리핑 60초)




In [1]:
# 환경 설정 및 라이브러리 설치
# - ffmpeg는 시스템 패키지: 미설치 시 `sudo apt-get install -y ffmpeg` 또는 `brew install ffmpeg`
!pip install -q langchain langchain-openai langchain-community pymupdf pillow pandas pydantic python-dotenv \
    opencv-python openai faiss-cpu



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import io
import base64
import time
import json
import subprocess
from pathlib import Path
from collections import Counter
from dotenv import load_dotenv

load_dotenv()

import fitz  # PyMuPDF
import cv2
import pandas as pd
from PIL import Image
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# 일부 문제는 Vision이 필요하지 않을 수 있음 — Vision 호출은 별도 시그니처로 명시
vision = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

DATA_DIR = Path("data")
PDF_REPORT = DATA_DIR / "quarterly_report.pdf"
PDF_PAPER = DATA_DIR / "research_paper.pdf"
AUDIO_DIR = DATA_DIR / "audio"
VIDEO_PATH = DATA_DIR / "video" / "earnings_briefing.mp4"

IMG_DIR = DATA_DIR / "_extracted"
FRAMES_DIR = DATA_DIR / "_frames"
AV_TMP_DIR = DATA_DIR / "_av_tmp"
for d in (IMG_DIR, FRAMES_DIR, AV_TMP_DIR):
    d.mkdir(parents=True, exist_ok=True)

assert PDF_REPORT.exists(), f"❌ {PDF_REPORT} 파일이 없습니다."
assert AUDIO_DIR.exists() and any(AUDIO_DIR.glob("*.mp3")), f"❌ {AUDIO_DIR}/*.mp3 없음."
assert VIDEO_PATH.exists(), f"❌ {VIDEO_PATH} 없음."

audio_files = sorted(AUDIO_DIR.glob("*.mp3"))
print("✅ 환경 설정 완료")
print(f"📄 PDF: {PDF_REPORT.name}, {PDF_PAPER.name}")
print(f"🎙️  Audio: {len(audio_files)}개 — {[f.name for f in audio_files]}")
print(f"🎬 Video: {VIDEO_PATH.name} ({VIDEO_PATH.stat().st_size:,} bytes)")


/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


✅ 환경 설정 완료
📄 PDF: quarterly_report.pdf, research_paper.pdf
🎙️  Audio: 3개 — ['meeting_hiring.mp3', 'meeting_product.mp3', 'meeting_revenue.mp3']
🎬 Video: earnings_briefing.mp4 (540,443 bytes)


---
## 📦 실습 데이터

| 카테고리 | 파일 | 내용 |
|----------|------|------|
| **PDF** | `quarterly_report.pdf` | 막대그래프, 라인차트, 실적표·KPI표 |
| **PDF** | `research_paper.pdf` | 파이프라인 그림, loss 그래프, 검색 성능표, 수식 |
| **Audio** | `audio/meeting_revenue.mp3` | 회의 발화 (매출) |
| **Audio** | `audio/meeting_product.mp3` | 회의 발화 (제품) |
| **Audio** | `audio/meeting_hiring.mp3` | 회의 발화 (채용) |
| **Video** | `video/earnings_briefing.mp4` | 실적 브리핑 영상 (PDF + 음성 합성) |

작업 디렉토리:
- `data/_extracted/` — PDF에서 뽑은 이미지
- `data/_frames/` — 비디오 프레임
- `data/_av_tmp/` — 비디오에서 분리한 오디오 등 임시


---
## 문제 1: PDF 페이지를 PNG 이미지로 렌더링

PyMuPDF로 PDF 각 페이지를 PNG로 변환하여 저장하는 함수를 작성하세요.

**요구사항:**
- 함수 시그니처: `render_pages_to_png(pdf_path: str, out_dir: str, dpi: int = 150) -> list[str]`
- 각 페이지를 `<out_dir>/<stem>_p<N>.png`로 저장
- 반환: 저장된 파일 경로 리스트

**평가기준:**
- 반환 길이 == 페이지 수
- 저장된 파일이 실제로 존재하고 PNG로 열림


In [3]:
# ✅ 문제 1 정답
def render_pages_to_png(pdf_path: str, out_dir: str, dpi: int = 150) -> list[str]:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    stem = Path(pdf_path).stem
    doc = fitz.open(pdf_path)
    paths = []
    for i, page in enumerate(doc, start=1):
        pix = page.get_pixmap(dpi=dpi)
        p = out_dir / f"{stem}_p{i}.png"
        pix.save(str(p))
        paths.append(str(p))
    doc.close()
    return paths


paths = render_pages_to_png(str(PDF_REPORT), str(IMG_DIR), dpi=120)
print(f"렌더된 페이지: {len(paths)}")
for p in paths:
    sz = Path(p).stat().st_size
    print(f"  {Path(p).name}: {sz:,} bytes")


렌더된 페이지: 2
  quarterly_report_p1.png: 106,239 bytes
  quarterly_report_p2.png: 90,478 bytes


---
## 문제 2: PDF 안에 임베드된 이미지 추출

PyMuPDF로 페이지에 삽입된 raster 이미지(차트, 그림)를 별도 파일로 추출하세요.

**요구사항:**
- 함수 시그니처: `extract_embedded_images(pdf_path: str, out_dir: str) -> list[dict]`
- 각 dict: `{"page", "index", "path", "width", "height"}`
- `page.get_images(full=True)` → `doc.extract_image(xref)` 사용
- 파일명: `<stem>_p<N>_img<I>.<ext>`

**평가기준:**
- 보고서 PDF에서 최소 2장 추출 (차트 2개)
- 각 dict에 5개 키 모두 존재
- 저장된 파일이 PIL.Image.open으로 열림


In [4]:
# ✅ 문제 2 정답
def extract_embedded_images(pdf_path: str, out_dir: str) -> list[dict]:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    stem = Path(pdf_path).stem
    doc = fitz.open(pdf_path)
    items = []
    for pi, page in enumerate(doc, start=1):
        for ii, info in enumerate(page.get_images(full=True), start=1):
            xref = info[0]
            data = doc.extract_image(xref)
            ext = data["ext"]
            path = out_dir / f"{stem}_p{pi}_img{ii}.{ext}"
            path.write_bytes(data["image"])
            items.append({
                "page": pi, "index": ii, "path": str(path),
                "width": data.get("width"), "height": data.get("height"),
            })
    doc.close()
    return items


items = extract_embedded_images(str(PDF_REPORT), str(IMG_DIR))
print(f"추출된 이미지: {len(items)}")
for it in items:
    print(f"  p{it['page']} idx{it['index']}: {Path(it['path']).name} ({it['width']}x{it['height']})")


추출된 이미지: 2
  p1 idx1: quarterly_report_p1_img1.png (520x320)
  p2 idx1: quarterly_report_p2_img1.png (520x320)


---
## 문제 3: GPT-4o Vision으로 이미지 캡션 생성

추출한 이미지를 base64로 인코딩하여 Vision LLM에 전달하고 한국어 캡션을 받아오세요.

**요구사항:**
- 함수 시그니처: `caption_image(image_path: str, prompt: str = "이 이미지를 한국어로 1~2문장으로 설명하세요.") -> str`
- `data:image/png;base64,...` URL 형태로 Vision에 전달
- `HumanMessage(content=[{"type":"text",...}, {"type":"image_url","image_url":{"url":...}}])`
- 반환: LLM 응답 텍스트

**평가기준:**
- `caption_image(path)`가 빈 문자열이 아니어야 함
- 캡션 길이 >= 10자


In [5]:
# ✅ 문제 3 정답
def caption_image(image_path: str, prompt: str = "이 이미지를 한국어로 1~2문장으로 설명하세요.") -> str:
    raw = Path(image_path).read_bytes()
    b64 = base64.b64encode(raw).decode()
    ext = Path(image_path).suffix.lstrip(".").lower()
    mime = "image/jpeg" if ext in ("jpg", "jpeg") else f"image/{ext}"
    msg = HumanMessage(content=[
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
    ])
    return vision.invoke([msg]).content


if items:
    cap = caption_image(items[0]["path"])
    print(f"📝 캡션: {cap}")


📝 캡션: 이 그래프는 2025년 분기별 수익을 나타내며, 각 분기(Q1, Q2, Q3, Q4)의 수익이 증가하는 추세를 보이고 있습니다. 특히 4분기(Q4)에서 가장 높은 수익인 188억 원을 기록했습니다.


---
## 문제 4: 차트 → Pydantic 구조 복원

막대그래프 이미지를 보고 Vision LLM이 카테고리와 값을 JSON으로 복원하도록 하세요. Pydantic으로 검증합니다.

**요구사항:**
- Pydantic 모델: `class ChartData(BaseModel): title: str; categories: list[str]; values: list[float]; unit: str`
- 함수 시그니처: `restore_chart(image_path: str) -> ChartData`
- Vision에 "JSON으로만 답하라" 지시 후 응답을 파싱하여 `ChartData(**parsed)` 반환
- LLM 응답에서 ``` 코드블록은 제거

**평가기준:**
- `categories`와 `values` 길이가 같음
- `len(values) >= 3`
- 모든 `values`가 float


In [6]:
# ✅ 문제 4 정답
class ChartData(BaseModel):
    title: str = Field(..., description="차트 제목")
    categories: list[str] = Field(..., description="x축 카테고리")
    values: list[float] = Field(..., description="y축 수치")
    unit: str = Field("", description="수치 단위 (예: 억원)")


def restore_chart(image_path: str) -> ChartData:
    raw = Path(image_path).read_bytes()
    b64 = base64.b64encode(raw).decode()
    ext = Path(image_path).suffix.lstrip(".").lower()
    mime = "image/jpeg" if ext in ("jpg", "jpeg") else f"image/{ext}"
    prompt = (
        '이 차트의 데이터를 JSON으로만 답하세요. 다른 텍스트 금지.\n'
        '키: title (제목), categories (문자열 배열), values (숫자 배열), unit (단위 문자열).\n'
        '예: {"title":"...", "categories":["Q1","Q2"], "values":[100, 120], "unit":"억원"}'
    )
    msg = HumanMessage(content=[
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
    ])
    text = vision.invoke([msg]).content.strip()
    if text.startswith("```"):
        text = text.strip("`").lstrip("json").strip()
    parsed = json.loads(text)
    return ChartData(**parsed)


if items:
    ch = restore_chart(items[0]["path"])
    print(f"📊 {ch.title}")
    for c, v in zip(ch.categories, ch.values):
        print(f"  {c}: {v} {ch.unit}")


📊 2025 Quarterly Revenue (억원)
  Q1: 120.0 억원
  Q2: 145.0 억원
  Q3: 162.0 억원
  Q4: 188.0 억원


---
## 문제 5: PyMuPDF로 표 영역 탐지

PyMuPDF 1.23+ 의 `page.find_tables()` API로 PDF 안의 표 위치를 찾는 함수를 작성하세요.

**요구사항:**
- 함수 시그니처: `detect_tables(pdf_path: str) -> list[dict]`
- 각 dict: `{"page", "bbox", "rows", "cols"}` — `bbox`는 `(x0, y0, x1, y1)` tuple
- `tables = page.find_tables()` → `tables.tables`

**평가기준:**
- 보고서 PDF에서 최소 1개 이상 표 검출
- bbox가 4-tuple


In [7]:
# ✅ 문제 5 정답
def detect_tables(pdf_path: str) -> list[dict]:
    doc = fitz.open(pdf_path)
    out = []
    for pi, page in enumerate(doc, start=1):
        try:
            tabs = page.find_tables()
        except Exception:
            continue
        for tb in tabs.tables:
            rows = len(tb.rows) if hasattr(tb, "rows") else 0
            cols = len(tb.header.names) if hasattr(tb, "header") and tb.header else 0
            out.append({
                "page": pi,
                "bbox": tuple(tb.bbox),
                "rows": rows,
                "cols": cols,
            })
    doc.close()
    return out


tabs = detect_tables(str(PDF_REPORT))
print(f"검출된 표: {len(tabs)}")
for t in tabs:
    print(f"  p{t['page']} bbox={tuple(round(v,1) for v in t['bbox'])} rows={t['rows']} cols={t['cols']}")


Consider using the pymupdf_layout package for a greatly improved page layout analysis.
검출된 표: 2
  p1 bbox=(99.2, 586.6, 496.1, 694.6) rows=6 cols=4
  p2 bbox=(104.9, 389.6, 490.4, 479.6) rows=5 cols=4


---
## 문제 6: PDF 표를 pandas DataFrame으로 추출

검출된 표를 `pandas.DataFrame`으로 변환하세요.

**요구사항:**
- 함수 시그니처: `extract_tables_as_dataframes(pdf_path: str) -> list[pd.DataFrame]`
- 각 표마다 `tb.to_pandas()` 사용 (PyMuPDF 내장)
- 결과 DataFrame은 빈 컬럼/행을 제거(`dropna(how="all")`)

**평가기준:**
- 보고서 PDF에서 최소 1개 DataFrame 반환
- 첫 DataFrame의 `shape[0] >= 2, shape[1] >= 2`
- 컬럼명이 헤더로 잡혀 있음(중복/None 허용)


In [8]:
# ✅ 문제 6 정답
def extract_tables_as_dataframes(pdf_path: str) -> list[pd.DataFrame]:
    doc = fitz.open(pdf_path)
    out = []
    for page in doc:
        try:
            tabs = page.find_tables()
        except Exception:
            continue
        for tb in tabs.tables:
            try:
                df = tb.to_pandas()
            except Exception:
                continue
            df = df.dropna(how="all").dropna(axis=1, how="all")
            if df.shape[0] >= 1 and df.shape[1] >= 2:
                out.append(df)
    doc.close()
    return out


dfs = extract_tables_as_dataframes(str(PDF_REPORT))
print(f"추출 표 수: {len(dfs)}")
for i, df in enumerate(dfs):
    print(f"\n=== Table {i+1} {df.shape} ===")
    print(df.head().to_string(index=False))


추출 표 수: 2

=== Table 1 (5, 4) ===
   사업부 매출(억) 영업이익(억) 전년동기 대비
AI 솔루션    98      16    +42%
  클라우드    54       6    +18%
   컨설팅    24       1     +5%
    기타    12       1     -2%
    합계   188      24    +35%

=== Table 2 (4, 4) ===
  지표 2024 Q4 2025 Q4    변화율
고객 수   1,250   1,820 +45.6%
 MAU     85만    132만 +55.3%
 NPS      42      58 +38.1%
 이탈률    3.2%    2.1% -34.4%


---
## 문제 7: 표 정제 — 숫자 컬럼 변환 + 결측 처리

문자열로 들어온 숫자 컬럼(예: `"+35%"`, `"188"`, `"1,820"`)을 float으로 변환하는 함수를 작성하세요.

**요구사항:**
- 함수 시그니처: `clean_numeric_columns(df: pd.DataFrame) -> pd.DataFrame`
- 콤마(`,`) 제거, 퍼센트(`%`) 제거, 단위(`억`, `만`) 제거, 음수기호(`-`) 보존
- 값의 70% 이상이 숫자로 변환 가능한 컬럼만 float으로 변환
- 변환 불가 값은 `NaN`으로

**평가기준:**
- `"1,820" → 1820.0`, `"+35%" → 35.0`
- 문자열만 있는 컬럼은 그대로 유지
- 원본 df는 변경되지 않음(불변성)


In [9]:
# ✅ 문제 7 정답
import re
_NUM = re.compile(r"[+]?\s*([-]?\d[\d,\.]*)")

def _to_num(x):
    if x is None:
        return float("nan")
    s = str(x).strip().replace(",", "").replace("%", "")
    for unit in ("억원", "억", "만원", "만", "원"):
        s = s.replace(unit, "")
    m = _NUM.search(s)
    if not m:
        return float("nan")
    try:
        return float(m.group(1).replace(",", ""))
    except ValueError:
        return float("nan")


def clean_numeric_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        if pd.api.types.is_numeric_dtype(out[col]):
            continue
        converted = out[col].apply(_to_num)
        success_rate = converted.notna().mean()
        if success_rate >= 0.7:
            out[col] = converted
    return out


if dfs:
    cleaned = clean_numeric_columns(dfs[0])
    print(cleaned.dtypes)
    print(cleaned.head())


사업부         object
매출(억)      float64
영업이익(억)    float64
전년동기 대비    float64
dtype: object
      사업부  매출(억)  영업이익(억)  전년동기 대비
0  AI 솔루션   98.0     16.0     42.0
1    클라우드   54.0      6.0     18.0
2     컨설팅   24.0      1.0      5.0
3      기타   12.0      1.0     -2.0
4      합계  188.0     24.0     35.0


---
## 문제 8: 표 → Markdown (LLM이 읽기 좋은 형식)

DataFrame을 Markdown 표 문자열로 변환하세요. RAG 컨텍스트에 표를 자연어처럼 끼워넣기 위해 사용합니다.

**요구사항:**
- 함수 시그니처: `df_to_markdown(df: pd.DataFrame, caption: str = "") -> str`
- 첫 줄에 `**caption**`(주어진 경우)
- `df.to_markdown(index=False)` 사용 (없으면 직접 구현 OK — `|` 구분, 헤더 아래 구분선)
- 결과는 `str` 타입

**평가기준:**
- 결과에 `|` 문자가 헤더+구분선 포함 최소 2회 등장
- `caption` 주어지면 `**caption**` 포함


In [10]:
# ✅ 문제 8 정답
def df_to_markdown(df: pd.DataFrame, caption: str = "") -> str:
    head = f"**{caption}**\n\n" if caption else ""
    try:
        body = df.to_markdown(index=False)
    except Exception:
        # 폴백: 직접 구성
        cols = list(df.columns)
        sep = "| " + " | ".join(["---"] * len(cols)) + " |"
        header = "| " + " | ".join(str(c) for c in cols) + " |"
        rows = ["| " + " | ".join(str(v) for v in row) + " |" for row in df.values.tolist()]
        body = "\n".join([header, sep] + rows)
    return head + body


if dfs:
    md_text = df_to_markdown(cleaned, caption="2025 Q4 사업부별 실적")
    print(md_text)


**2025 Q4 사업부별 실적**

| 사업부    |   매출(억) |   영업이익(억) |   전년동기 대비 |
|:----------|-----------:|---------------:|----------------:|
| AI 솔루션 |         98 |             16 |              42 |
| 클라우드  |         54 |              6 |              18 |
| 컨설팅    |         24 |              1 |               5 |
| 기타      |         12 |              1 |              -2 |
| 합계      |        188 |             24 |              35 |


---
## 문제 9: Vision OCR — 페이지 이미지에서 텍스트 추출

`render_pages_to_png`로 만든 페이지 이미지를 GPT-4o Vision에 보내 텍스트를 OCR하세요.

**요구사항:**
- 함수 시그니처: `vision_ocr(image_path: str) -> str`
- 프롬프트: "이 이미지의 모든 텍스트를 원본 순서대로 추출하세요. 표는 행/열 구조를 보존하세요. 해석/요약 금지."
- 반환: OCR 텍스트 (한국어 보존)

**평가기준:**
- 첫 페이지 OCR 결과에 "Modu Tech" 또는 "분기" 류 키워드 포함
- 빈 문자열 아님


In [11]:
# ✅ 문제 9 정답
def vision_ocr(image_path: str) -> str:
    raw = Path(image_path).read_bytes()
    b64 = base64.b64encode(raw).decode()
    ext = Path(image_path).suffix.lstrip(".").lower()
    mime = "image/jpeg" if ext in ("jpg", "jpeg") else f"image/{ext}"
    prompt = ("이 이미지의 모든 텍스트를 원본 순서대로 추출하세요. "
              "표는 행/열 구조를 보존하세요. 해석/요약 금지.")
    msg = HumanMessage(content=[
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
    ])
    return vision.invoke([msg]).content


ocr_text = vision_ocr(paths[0])
print(f"OCR 결과(앞 300자):\n{ocr_text[:300]}")


OCR 결과(앞 300자):
```
Modu Tech - Confidential

Modu Tech 2025년 4분기 실적 보고서
발행: 2026년 1월 30일 · IR팀

1. 요약
당사는 2025년 4분기 매출 188억원, 영업이익 24억원을 기록하며 전년 동기 대비 매출 35% 성장을 달성했습니다. 실 솔루션 부문이 견조한 성장세를 보이며 전체 매출 증가를 견인했습니다. 본 보고서는 분기 실적과 함께 사업부별 성과, 주요 지표, 향후 전망을 포함합니다.

2. 분기별 매출 추이
2025 Quarterly Revenue (억원)

그림 1. 2025년 분기


---
## 문제 10: Whisper로 오디오 파일 → 텍스트 추출

LangChain의 `OpenAIWhisperParser`로 회의 mp3를 텍스트화하세요.

**요구사항:**
- 함수 시그니처: `transcribe_audio(audio_path: str) -> dict`
- LangChain 패턴: `OpenAIWhisperParser().lazy_parse(Blob.from_path(path))` → `list[Document]`
- 반환: `{"source": str, "text": str, "n_chunks": int, "duration_sec": float}`
- `duration_sec`은 `cv2.VideoCapture(path)` 또는 `mutagen` 없이도 OK — 안 되면 0.0
- 25MB 넘는 파일은 자동으로 여러 chunk로 잘림 → text 합쳐서 반환

**평가기준:**
- `text` 길이 >= 30자 (한국어 회의 발화)
- `source`는 파일명만 (경로 X)
- 4개 키 모두 존재


In [12]:
# ✅ 문제 10 정답
from langchain_community.document_loaders.parsers.audio import OpenAIWhisperParser
from langchain_core.document_loaders.blob_loaders import Blob


def transcribe_audio(audio_path: str) -> dict:
    parser = OpenAIWhisperParser()
    blob = Blob.from_path(audio_path)
    docs = list(parser.lazy_parse(blob))
    text = " ".join(d.page_content for d in docs).strip()

    duration = 0.0
    try:
        cap = cv2.VideoCapture(audio_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        n = cap.get(cv2.CAP_PROP_FRAME_COUNT)
        if fps and n:
            duration = n / fps
        cap.release()
    except Exception:
        pass

    return {
        "source": Path(audio_path).name,
        "text": text,
        "n_chunks": len(docs),
        "duration_sec": round(duration, 2),
    }


sample_audio = audio_files[0]
result = transcribe_audio(str(sample_audio))
print(f"📁 {result['source']}  ({result['duration_sec']:.1f}s, {result['n_chunks']} chunk)")
print(f"📝 {result['text'][:200]}...")


/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Transcribing part 1!
📁 meeting_hiring.mp3  (0.0s, 1 chunk)
📝 인사팀에서 채용 계획을 공유합니다. 올해 상반기에 AI 엔지니어 2명, 백엔드 개발자 15명, 데이터 분석가 10명을 채용할 예정입니다. 특히 AI 엔지니어는 멀티모달 ARG 경험자를 우대합니다. 신규 입사자 옴보딩은 4주 과정으로 진행되며 한국어와 영어 두 언어로 제공됩니다. 지원자는 회사 홈페이지의 채용 페이지를 통해 지원해 주시기 바랍니다....


---
## 문제 11: 오디오 transcript → 청크 Document 리스트

문제 10에서 받은 transcript를 RAG가 검색하기 좋게 청크로 나누세요. `RecursiveCharacterTextSplitter` 사용.

**요구사항:**
- 함수 시그니처: `chunk_audio_transcript(transcript: dict, chunk_size: int = 300, chunk_overlap: int = 50) -> list[Document]`
- `LangChain`의 `RecursiveCharacterTextSplitter`로 `transcript["text"]` 분할
- 각 Document `metadata`: `{"source", "element_type": "audio_chunk", "chunk_index"}`
- `source`는 transcript에서 그대로 가져오기
- 빈 청크는 제외

**평가기준:**
- 반환 길이 >= 1
- 모든 Document에 3개 메타 키 존재
- `chunk_index`가 0부터 1씩 증가
- 모든 `page_content`가 빈 문자열 아님


In [13]:
# ✅ 문제 11 정답
from langchain_text_splitters import RecursiveCharacterTextSplitter


def chunk_audio_transcript(transcript: dict, chunk_size: int = 300, chunk_overlap: int = 50) -> list[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_text(transcript["text"])
    docs = []
    for i, c in enumerate(chunks):
        if not c.strip():
            continue
        docs.append(Document(
            page_content=c,
            metadata={
                "source": transcript["source"],
                "element_type": "audio_chunk",
                "chunk_index": i,
            },
        ))
    return docs


audio_docs = chunk_audio_transcript(result, chunk_size=200, chunk_overlap=40)
print(f"오디오 청크: {len(audio_docs)}개")
for d in audio_docs[:3]:
    print(f"  [{d.metadata['element_type']}] #{d.metadata['chunk_index']}  {d.page_content[:80]!r}")


오디오 청크: 1개
  [audio_chunk] #0  '인사팀에서 채용 계획을 공유합니다. 올해 상반기에 AI 엔지니어 2명, 백엔드 개발자 15명, 데이터 분석가 10명을 채용할 예정입니다. 특히 '


---
## 문제 12: 비디오 프레임 샘플링 + Vision 캡션 → Document

비디오에서 매 N초마다 프레임을 뽑아 GPT-4o Vision으로 캡션을 만들고 Document로 묶으세요.

**요구사항:**
- 함수 시그니처: `caption_video_frames(video_path: str, out_dir: str, every_sec: int = 10) -> list[Document]`
- `cv2.VideoCapture` + `cap.set(cv2.CAP_PROP_POS_MSEC, sec*1000)` + `cap.read()`로 시각 점프 후 frame 추출
- 각 frame을 `<out_dir>/<stem>_t<sec>s.png`로 저장
- `caption_image()` (문제 3에서 만든 함수) 재사용해 캡션 생성
- Document `metadata`: `{"source", "element_type": "video_frame_caption", "timestamp_sec"}`
- 마지막 프레임까지 자연스럽게 끝나야 함 (`read()`가 False면 종료)

**평가기준:**
- Document 1개 이상 (60초 영상 × every_sec=10 → ~6개 기대)
- 모든 `page_content`가 빈 문자열 아님 (캡션 생성됨)
- `timestamp_sec`이 정수, 오름차순
- 저장된 png 파일이 실제로 존재


In [14]:
# ✅ 문제 12 정답
def caption_video_frames(video_path: str, out_dir: str, every_sec: int = 10) -> list[Document]:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    stem = Path(video_path).stem
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = (n_frames / fps) if fps else 0
    docs: list[Document] = []
    sec = 0
    while sec <= duration:
        cap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000)
        ok, frame = cap.read()
        if not ok:
            break
        path = out_dir / f"{stem}_t{sec:03d}s.png"
        cv2.imwrite(str(path), frame)
        caption = caption_image(str(path))
        docs.append(Document(
            page_content=caption,
            metadata={
                "source": Path(video_path).name,
                "element_type": "video_frame_caption",
                "timestamp_sec": sec,
            },
        ))
        sec += every_sec
    cap.release()
    return docs


video_docs = caption_video_frames(str(VIDEO_PATH), str(FRAMES_DIR), every_sec=15)
print(f"비디오 프레임 Document: {len(video_docs)}개")
for d in video_docs:
    print(f"  t={d.metadata['timestamp_sec']:>3}s  {d.page_content[:80]}")


비디오 프레임 Document: 2개
  t=  0s  이 문서는 Modu Tech의 2025년 4분기 실적 보고서로, 매출 188억 원과 영업이익 24억 원을 기록하며 전년 동기 대비 각각 35% 
  t= 15s  이 이미지는 월별 매출 추이와 시장 평균을 비교한 그래프와 주요 KPI를 정리한 표를 포함하고 있습니다. 2026년까지 AI 솔루션 부문 매출 


---
## 문제 13: 비디오의 오디오 트랙 분리 + Whisper transcribe

비디오에서 오디오 트랙만 추출(`ffmpeg`)한 뒤 문제 10의 `transcribe_audio`를 재사용하세요.

**요구사항:**
- 함수 시그니처: `transcribe_video_audio(video_path: str, audio_out_dir: str) -> dict`
- 출력 mp3: `<audio_out_dir>/<stem>.mp3`
- ffmpeg 명령: `ffmpeg -y -i <video> -vn -acodec libmp3lame <out.mp3>` (또는 `-acodec copy` 후 컨테이너만 변경 — 영상에 mp3 트랙이 있다면)
- `subprocess.run(..., check=True, capture_output=True)` 권장
- 추출된 mp3를 `transcribe_audio()`에 넣어 결과 반환
- 반환 dict에 `"video_source": <video filename>` 키 추가

**평가기준:**
- 반환 dict에 `text`, `source`, `video_source` 키 존재
- `text` 길이 >= 30자
- mp3 파일이 디스크에 실제로 생성됨

> 💡 **시스템 ffmpeg 확인**: `!ffmpeg -version` 으로 사전 체크.


In [15]:
# ✅ 문제 13 정답
def transcribe_video_audio(video_path: str, audio_out_dir: str) -> dict:
    audio_out_dir = Path(audio_out_dir)
    audio_out_dir.mkdir(parents=True, exist_ok=True)
    audio_path = audio_out_dir / (Path(video_path).stem + ".mp3")
    subprocess.run(
        ["ffmpeg", "-y", "-i", video_path, "-vn", "-acodec", "libmp3lame", str(audio_path)],
        check=True, capture_output=True,
    )
    result = transcribe_audio(str(audio_path))
    result["video_source"] = Path(video_path).name
    return result


va_result = transcribe_video_audio(str(VIDEO_PATH), str(AV_TMP_DIR))
print(f"🎬 video: {va_result['video_source']}")
print(f"🎙️  extracted: {va_result['source']}  ({va_result.get('duration_sec', 0):.1f}s)")
print(f"📝 {va_result['text'][:200]}...")


Transcribing part 1!
🎬 video: earnings_briefing.mp4
🎙️  extracted: earnings_briefing.mp3  (0.0s)
📝 오늘 분기 실적 회의를 시작하겠습니다. 2025년 4분기 매출은 188억 원으로 전 분기 대비 16% 증가했습니다. 특히 AI 솔루션 사업부가 98억 원의 매출을 기록하며 가장 큰 성장을 보였습니다. 영업이익은 24억 원으로 전년 동기 대비 35% 늘었습니다. 다음 분기는 동남아시아 시장 진출 본격화할 계획입니다....


---
## 문제 14: PDF + 오디오 + 비디오를 하나의 `Document` 리스트로 통합 🏁

문제 1~13의 결과를 단일 `Document` 리스트로 묶는 빌더를 작성하세요. Weekend 3 멀티모달 RAG의 직접 입력이 됩니다.

**요구사항:**
- 함수 시그니처:
  ```python
  build_multimodal_documents(
      pdf_path: str,
      img_dir: str,
      audio_paths: list[str] | None = None,
      video_path: str | None = None,
      av_tmp_dir: str | None = None,
  ) -> list[Document]
  ```
- 각 Document의 `metadata`는 모달리티에 따라 다름:
  - 표: `{"source", "page_number", "element_type": "table"}`
  - 이미지: `{"source", "page_number", "element_type": "image_caption"}`
  - 페이지 OCR: `{"source", "page_number", "element_type": "page_ocr"}`
  - 오디오 청크: `{"source", "element_type": "audio_chunk", "chunk_index"}`
  - 비디오 프레임: `{"source", "element_type": "video_frame_caption", "timestamp_sec"}`
  - 비디오 오디오 청크: `{"source", "element_type": "video_audio_chunk", "chunk_index"}`
- 속도용: PDF는 첫 2장 이미지 + 첫 1페이지 OCR만, 비디오는 `every_sec=15`로 OK
- `audio_paths=None` / `video_path=None`이면 해당 모달리티 스킵 (PDF 만 처리 가능)

**평가기준:**
- `pdf_path` + `audio_paths` + `video_path` 모두 주어졌을 때:
  - 반환 길이 >= 6
  - `set(d.metadata["element_type"] for d in docs)` ⊇ {`"table"`, `"image_caption"`, `"page_ocr"`, `"audio_chunk"`, `"video_frame_caption"`, `"video_audio_chunk"`} (모두)
  - 모든 Document에 `source`, `element_type` 키 존재
- `audio_paths=None, video_path=None`일 때: 문제 10(이전 버전)과 동일하게 PDF만 처리


In [16]:
# ✅ 문제 14 정답
def build_multimodal_documents(
    pdf_path: str,
    img_dir: str,
    audio_paths: list | None = None,
    video_path: str | None = None,
    av_tmp_dir: str | None = None,
) -> list[Document]:
    docs: list[Document] = []
    pdf_src = Path(pdf_path).name

    # 1) 표
    doc = fitz.open(pdf_path)
    for pi, page in enumerate(doc, start=1):
        try:
            tabs = page.find_tables()
        except Exception:
            continue
        for tb in tabs.tables:
            try:
                df = tb.to_pandas().dropna(how="all").dropna(axis=1, how="all")
            except Exception:
                continue
            if df.shape[0] < 1 or df.shape[1] < 2:
                continue
            md_text = df_to_markdown(clean_numeric_columns(df), caption=f"Table p{pi}")
            docs.append(Document(page_content=md_text,
                                 metadata={"source": pdf_src, "page_number": pi, "element_type": "table"}))
    doc.close()

    # 2) PDF 이미지 캡션 (첫 2장)
    for it in extract_embedded_images(pdf_path, img_dir)[:2]:
        cap = caption_image(it["path"])
        docs.append(Document(page_content=cap,
                             metadata={"source": pdf_src, "page_number": it["page"],
                                       "element_type": "image_caption"}))

    # 3) 첫 페이지 OCR
    rendered = render_pages_to_png(pdf_path, img_dir, dpi=120)
    if rendered:
        ocr = vision_ocr(rendered[0])
        docs.append(Document(page_content=ocr,
                             metadata={"source": pdf_src, "page_number": 1,
                                       "element_type": "page_ocr"}))

    # 4) 오디오
    for ap in (audio_paths or []):
        tr = transcribe_audio(ap)
        docs.extend(chunk_audio_transcript(tr, chunk_size=300, chunk_overlap=50))

    # 5) 비디오 (프레임 캡션 + 오디오 트랙 transcribe)
    if video_path:
        docs.extend(caption_video_frames(video_path, str(Path(av_tmp_dir or img_dir) / "frames"),
                                          every_sec=15))
        if av_tmp_dir:
            va_tr = transcribe_video_audio(video_path, av_tmp_dir)
            va_chunks = chunk_audio_transcript(va_tr, chunk_size=300, chunk_overlap=50)
            for d in va_chunks:
                d.metadata["element_type"] = "video_audio_chunk"
                d.metadata["source"] = va_tr["video_source"]
            docs.extend(va_chunks)
    return docs


mm_docs = build_multimodal_documents(
    pdf_path=str(PDF_REPORT),
    img_dir=str(IMG_DIR),
    audio_paths=[str(audio_files[0])],
    video_path=str(VIDEO_PATH),
    av_tmp_dir=str(AV_TMP_DIR),
)
print(f"총 멀티모달 Documents: {len(mm_docs)}개")
print(f"\n모달리티별 분포:")
counter = Counter(d.metadata["element_type"] for d in mm_docs)
for k, v in counter.most_common():
    print(f"  {k:<22} {v}")


Transcribing part 1!
Transcribing part 1!
총 멀티모달 Documents: 9개

모달리티별 분포:
  table                  2
  image_caption          2
  video_frame_caption    2
  page_ocr               1
  audio_chunk            1
  video_audio_chunk      1


---
## 문제 15: 멀티모달 Document를 단일 FAISS 인덱스에 통합

문제 14에서 만든 `Document` 리스트를 통째로 FAISS에 임베딩·인덱싱하세요. 표·이미지·오디오·비디오가 *같은 벡터 공간*에 들어갑니다.

**요구사항:**
- 함수 시그니처: `build_faiss_index(docs: list[Document]) -> FAISS`
- `OpenAIEmbeddings(model="text-embedding-3-small")` (이미 `embeddings` 객체로 셋업됨)
- `FAISS.from_documents(docs, embeddings)` 패턴
- `metadata`는 자동 보존됨 (`element_type` 검색 시 활용)
- 빈 docs 입력 시 `ValueError` raise

**평가기준:**
- 반환 객체가 `FAISS` 타입
- `vs.similarity_search("매출")` 호출 시 `Document` 리스트 반환
- 반환된 Document들이 원본 `metadata["element_type"]`을 유지


In [17]:
# ✅ 문제 15 정답
def build_faiss_index(docs: list[Document]) -> FAISS:
    if not docs:
        raise ValueError("empty docs")
    return FAISS.from_documents(docs, embeddings)


vs = build_faiss_index(mm_docs)
hits = vs.similarity_search("Modu Tech 매출", k=4)
print(f"검색 결과 {len(hits)}개:")
for h in hits:
    et = h.metadata.get("element_type", "?")
    print(f"  [{et:<22}] {h.page_content[:70]}")


검색 결과 4개:
  [video_frame_caption   ] 이 보고서는 Modu Tech의 2025년 4분기 실적을 다루고 있으며, 매출 188억 원과 영업이익 24억 원을 기록하여 전
  [page_ocr              ] ```
Modu Tech - Confidential

Modu Tech 2025년 4분기 실적 보고서
발행: 2026년 1월 
  [video_audio_chunk     ] 오늘 분기 실적 회의를 시작하겠습니다. 2025년 4분기 매출은 188억 원으로 전 분기 대비 16% 증가했습니다. 특히 AI
  [table                 ] **Table p1**

| 사업부    |   매출(억) |   영업이익(억) |   전년동기 대비 |
|:---------


---
## 문제 16: `element_type` 별 가중치 검색

같은 유사도라도 표는 더 신뢰하고 이미지 캡션은 덜 신뢰하고 싶을 때. `similarity_search_with_score`로 점수 받아 가중치 곱해 재정렬.

**요구사항:**
- 함수 시그니처:
  ```python
  weighted_search(vs: FAISS, query: str,
                  weights: dict[str, float] | None = None,
                  k: int = 5, fetch_k: int = 20) -> list[tuple[Document, float]]
  ```
- 기본 `weights`:
  ```python
  {"table": 1.4, "page_ocr": 1.0, "image_caption": 0.8,
   "audio_chunk": 1.1, "video_frame_caption": 0.9, "video_audio_chunk": 1.1}
  ```
- 동작:
  1. `similarity_search_with_score(query, k=fetch_k)` — FAISS는 *거리*(낮을수록 좋음) 반환
  2. `score = weights.get(element_type, 1.0) / (distance + 1e-9)` — *높을수록 좋음*으로 변환 + 가중
  3. score 내림차순 정렬 → 상위 k개 반환

**평가기준:**
- 반환이 `[(Document, float)]` 리스트
- 가중치 0 주면 그 element_type은 결과에서 사실상 사라짐
- 기본 가중치 사용 시 표(table)가 같은 거리의 image_caption보다 위에 옴


In [18]:
# ✅ 문제 16 정답
DEFAULT_WEIGHTS = {
    "table": 1.4,
    "page_ocr": 1.0,
    "image_caption": 0.8,
    "audio_chunk": 1.1,
    "video_frame_caption": 0.9,
    "video_audio_chunk": 1.1,
}


def weighted_search(vs: FAISS, query: str,
                    weights: dict | None = None,
                    k: int = 5, fetch_k: int = 20) -> list[tuple]:
    w = DEFAULT_WEIGHTS if weights is None else weights
    hits = vs.similarity_search_with_score(query, k=fetch_k)
    scored = []
    for doc, distance in hits:
        et = doc.metadata.get("element_type", "")
        weight = w.get(et, 1.0)
        if weight == 0:
            continue
        score = weight / (distance + 1e-9)
        scored.append((doc, score))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:k]


print("🔍 'Q4 매출 차트' — 기본 가중치 (표 우대)")
for doc, sc in weighted_search(vs, "Q4 매출 차트", k=5):
    et = doc.metadata.get("element_type", "?")
    print(f"  {sc:.3f}  [{et:<22}] {doc.page_content[:60]}")

print("\n🔍 같은 쿼리 — 이미지 caption만 (다른 모달 0 가중치)")
zeros = {"table": 0, "page_ocr": 0, "audio_chunk": 0, "video_audio_chunk": 0}
for doc, sc in weighted_search(vs, "Q4 매출 차트", weights=zeros, k=3):
    et = doc.metadata.get("element_type", "?")
    print(f"  {sc:.3f}  [{et:<22}] {doc.page_content[:60]}")


🔍 'Q4 매출 차트' — 기본 가중치 (표 우대)
  1.133  [table                 ] **Table p1**

| 사업부    |   매출(억) |   영업이익(억) |   전년동기 대비 |
|
  1.127  [table                 ] **Table p2**

| 지표    |   2024 Q4 |   2025 Q4 |   변화율 |
|:--
  0.780  [video_audio_chunk     ] 오늘 분기 실적 회의를 시작하겠습니다. 2025년 4분기 매출은 188억 원으로 전 분기 대비 16% 증가했
  0.763  [page_ocr              ] ```
Modu Tech - Confidential

Modu Tech 2025년 4분기 실적 보고서
발행:
  0.743  [image_caption         ] 이 그래프는 2025년 분기별 수익을 나타내며, 각 분기(Q1, Q2, Q3, Q4)의 수익이 억원 단위로 

🔍 같은 쿼리 — 이미지 caption만 (다른 모달 0 가중치)
  0.929  [image_caption         ] 이 그래프는 2025년 분기별 수익을 나타내며, 각 분기(Q1, Q2, Q3, Q4)의 수익이 억원 단위로 
  0.777  [image_caption         ] 이 그래프는 1월부터 6월까지의 월별 매출 추이와 시장 평균을 비교한 것입니다. 매출은 전반적으로 증가하는 
  0.748  [video_frame_caption   ] 이 이미지는 월별 매출 추이와 시장 평균을 비교한 그래프와 주요 KPI(핵심 성과 지표) 데이터를 포함하고 


---
## 문제 17: 멀티모달 컨텍스트로 RAG 답변 생성

가중치 검색 결과를 LLM 프롬프트에 *모달리티 별로 묶어* 넣고 답변을 만드세요. 모달리티별로 컨텍스트가 어떻게 다른지 LLM이 인지하도록.

**요구사항:**
- 함수 시그니처:
  ```python
  multimodal_rag_answer(vs: FAISS, query: str, k: int = 6) -> dict
  ```
- 동작:
  1. `weighted_search(vs, query, k=k)`로 컨텍스트 수집
  2. 컨텍스트를 `element_type`별로 그룹핑해서 프롬프트 섹션으로 정리
     - 예: `### 표 (PDF)\n<content>` / `### 이미지 캡션\n<content>` / `### 회의 발화\n<content>` / `### 비디오 프레임\n<content>`
  3. 시스템 메시지: "각 섹션은 다른 모달리티에서 추출되었음. 표는 정량 정보, 발화는 발언 내용, 프레임은 시각 정보."
  4. `llm.invoke([SystemMessage, HumanMessage])`로 답변 생성
- 반환: `{"answer": str, "sources": list[dict]}` — `sources`는 각 컨텍스트의 `{"element_type", "snippet": page_content[:80]}`

**평가기준:**
- 반환 dict에 `answer`, `sources` 키 존재
- `len(sources) <= k`
- `answer` 길이 >= 20자


In [19]:
# ✅ 문제 17 정답
ELEMENT_LABELS = {
    "table":               "표 (PDF)",
    "page_ocr":            "페이지 텍스트 (PDF OCR)",
    "image_caption":       "이미지 캡션",
    "audio_chunk":         "회의 발화",
    "video_frame_caption": "비디오 프레임",
    "video_audio_chunk":   "비디오 발화",
}


def multimodal_rag_answer(vs: FAISS, query: str, k: int = 6) -> dict:
    hits = weighted_search(vs, query, k=k)
    # 그룹핑
    groups: dict[str, list[str]] = {}
    sources = []
    for doc, _ in hits:
        et = doc.metadata.get("element_type", "etc")
        groups.setdefault(et, []).append(doc.page_content)
        sources.append({"element_type": et, "snippet": doc.page_content[:80]})

    # 섹션 텍스트
    sections = []
    for et, contents in groups.items():
        label = ELEMENT_LABELS.get(et, et)
        body = "\n".join(f"- {c}" for c in contents)
        sections.append(f"### {label}\n{body}")
    ctx = "\n\n".join(sections)

    sys = SystemMessage(content=(
        "당신은 멀티모달 RAG 어시스턴트입니다. 컨텍스트의 각 섹션은 서로 다른 모달리티에서 추출됐습니다.\n"
        "- 표(PDF): 정량 정보, 수치 비교에 우선\n"
        "- 이미지 캡션: 시각 정보의 자연어 요약\n"
        "- 회의 발화 / 비디오 발화: 발언자의 의도와 결정\n"
        "- 비디오 프레임: 동영상의 시각 정보\n"
        "근거 부족 시 '제공된 자료에서는 확인 불가'라고 답하세요."
    ))
    user = HumanMessage(content=f"질문: {query}\n\n컨텍스트:\n{ctx}")
    answer = llm.invoke([sys, user]).content
    return {"answer": answer, "sources": sources}


r = multimodal_rag_answer(vs, "Modu Tech의 Q4 매출과 회의에서 언급된 주요 전략은?", k=6)
print("=" * 70)
print(r["answer"])
print("=" * 70)
print(f"\n출처 {len(r['sources'])}개:")
for s in r["sources"]:
    print(f"  [{s['element_type']:<22}] {s['snippet']!r}")


Modu Tech의 2025년 4분기 매출은 188억 원이며, 영업이익은 24억 원으로 전년 동기 대비 각각 35% 성장했습니다. 주요 전략으로는 AI 솔루션 부문이 가장 큰 성장 동력으로 작용했으며, 다음 분기에는 동남아시아 시장 진출을 본격화할 계획입니다.

출처 6개:
  [video_frame_caption   ] '이 보고서는 Modu Tech의 2025년 4분기 실적을 다루고 있으며, 매출 188억 원과 영업이익 24억 원을 기록하여 전년 동기 대비 각각'
  [page_ocr              ] '```\nModu Tech - Confidential\n\nModu Tech 2025년 4분기 실적 보고서\n발행: 2026년 1월 30일 · IR팀\n'
  [table                 ] '**Table p2**\n\n| 지표    |   2024 Q4 |   2025 Q4 |   변화율 |\n|:--------|----------:|-'
  [table                 ] '**Table p1**\n\n| 사업부    |   매출(억) |   영업이익(억) |   전년동기 대비 |\n|:----------|--------'
  [video_audio_chunk     ] '오늘 분기 실적 회의를 시작하겠습니다. 2025년 4분기 매출은 188억 원으로 전 분기 대비 16% 증가했습니다. 특히 AI 솔루션 사업부가 '
  [video_frame_caption   ] '이 이미지는 월별 매출 추이와 시장 평균을 비교한 그래프와 주요 KPI(핵심 성과 지표) 데이터를 포함하고 있습니다. 2026년까지 AI 솔루션'


---
## 문제 18: LLM-as-Judge — 답변 충실도/관련성 평가 🏁

문제 17의 답변이 *주어진 컨텍스트에 충실한지* 다른 LLM으로 평가하세요. RAG 운영 시 hallucination 모니터링의 기본 패턴.

**요구사항:**
- 함수 시그니처:
  ```python
  judge_answer(query: str, answer: str, sources: list[dict]) -> dict
  ```
- Pydantic 스키마:
  ```python
  class JudgeVerdict(BaseModel):
      faithfulness: int = Field(..., ge=1, le=5, description="컨텍스트만 보고 정당화 가능한가")
      relevance:    int = Field(..., ge=1, le=5, description="질문에 실제로 답하는가")
      reasoning:    str = Field(..., description="짧은 근거(한국어)")
  ```
- 동작:
  1. `llm.with_structured_output(JudgeVerdict)`로 구조화 출력 강제
  2. 시스템: "당신은 RAG 답변 평가자. faithfulness=5는 컨텍스트로 완전히 정당화. relevance=5는 질문에 직접 답."
  3. 유저: 질문/답변/sources(각 element_type + snippet)을 보여줌
  4. 반환: `verdict.model_dump()`

**평가기준:**
- 반환 dict에 `faithfulness`, `relevance`, `reasoning` 3개 키
- 정수 범위 1~5
- `reasoning`이 빈 문자열 아님

**전체 파이프라인 통합 — 마지막 검증**:
```
PDF/Audio/Video → Document (#14) → FAISS (#15)
              → 가중치 검색 (#16) → 멀티모달 답변 (#17) → Judge 평가 (#18)
```
Weekend 3에서는 이 답변에 **Knowledge Graph 이웃 정보**를 추가로 보강(Graph-RAG)합니다.


In [20]:
# ✅ 문제 18 정답
class JudgeVerdict(BaseModel):
    faithfulness: int = Field(..., ge=1, le=5, description="컨텍스트만 보고 정당화 가능한가")
    relevance:    int = Field(..., ge=1, le=5, description="질문에 실제로 답하는가")
    reasoning:    str = Field(..., description="짧은 근거(한국어)")


def judge_answer(query: str, answer: str, sources: list[dict]) -> dict:
    judge_llm = llm.with_structured_output(JudgeVerdict)
    src_text = "\n".join(f"- [{s['element_type']}] {s['snippet']}" for s in sources)
    sys = SystemMessage(content=(
        "당신은 RAG 답변 평가자입니다.\n"
        "- faithfulness=5: 답변의 모든 주장이 컨텍스트로 정당화됨\n"
        "- faithfulness=1: 컨텍스트에 없는 정보를 만들어냄(hallucination)\n"
        "- relevance=5: 질문에 직접 답함\n"
        "- relevance=1: 질문과 무관"
    ))
    user = HumanMessage(content=(
        f"질문: {query}\n\n답변: {answer}\n\n사용된 컨텍스트:\n{src_text}"
    ))
    verdict = judge_llm.invoke([sys, user])
    return verdict.model_dump()


query = "Modu Tech의 Q4 매출과 회의에서 언급된 주요 전략은?"
r = multimodal_rag_answer(vs, query, k=6)
verdict = judge_answer(query, r["answer"], r["sources"])

print("=" * 70)
print(f"질문: {query}")
print(f"답변: {r['answer']}")
print("=" * 70)
print(f"⚖️  Faithfulness: {verdict['faithfulness']}/5")
print(f"⚖️  Relevance:    {verdict['relevance']}/5")
print(f"💭 Reasoning:    {verdict['reasoning']}")


질문: Modu Tech의 Q4 매출과 회의에서 언급된 주요 전략은?
답변: Modu Tech의 2025년 4분기 매출은 188억 원이며, 영업이익은 24억 원으로 전년 동기 대비 각각 35% 성장했습니다. 주요 전략으로는 AI 솔루션 부문이 가장 큰 성장 동력으로 작용했으며, 다음 분기에는 동남아시아 시장 진출을 본격화할 계획입니다.
⚖️  Faithfulness: 5/5
⚖️  Relevance:    5/5
💭 Reasoning:    답변은 Modu Tech의 2025년 4분기 매출과 영업이익에 대한 정보를 정확하게 제공하며, 전년 동기 대비 성장률도 언급하고 있습니다. 또한, 주요 전략으로 AI 솔루션 부문과 동남아시아 시장 진출 계획을 포함하고 있어 질문에 대한 답변이 충실합니다.


/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=JudgeVerdict(faithfulness...이 충실합니다.'), input_type=JudgeVerdict])
  return self.__pydantic_serializer__.to_python(
